In [24]:
from os import path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from datetime import datetime
from pathlib import PureWindowsPath
from typing import Dict

In [27]:


def build_filenames_and_urls(ts: datetime, province: str = "广东") -> Dict[str, Dict[str, str]]:
    """
    根据指定时间 ts（datetime），返回：
      1) 国家站：文件名与完整 Windows 共享路径
      2) 区域站：文件名与完整 Windows 共享路径
      3) 国家局5km能见度实况融合产品：文件名与完整 URL

    参数
    ----
    ts : datetime
        目标时间。国家站/区域站精确到分钟；能见度产品按小时。
    province : str
        省份名称（文件名中的中文省份名），默认 "广东"。

    返回
    ----
    dict，结构如下：
    {
        "national": {"filename": str, "fullpath": str},
        "regional": {"filename": str, "fullpath": str},
        "visibility": {"filename": str, "url": str},
    }
    """
    YYYY = ts.strftime("%Y")
    MM   = ts.strftime("%m")
    DD   = ts.strftime("%d")
    HH   = ts.strftime("%H")
    mm   = ts.strftime("%M")

    # 国家站
    national_filename = f"SurfAuto_{province}_{YYYY}{MM}{DD}{HH}{mm}00.csv"
    national_fullpath = str(
        PureWindowsPath(r"\\10.148.44.81\surf\idea\getSurfAutoOrg4Prov") / YYYY / MM / national_filename
    )

    # 区域站
    regional_filename = f"SurfAwst_{province}_{YYYY}{MM}{DD}{HH}{mm}00.csv"
    regional_fullpath = str(
        PureWindowsPath(r"\\10.148.44.81\surf\idea\getSurfAwstOrg4Prov") / YYYY / MM / regional_filename
    )

    # 国家局5km能见度实况融合产品（按小时）
    vis_dir_yyyymmdd = f"{YYYY}{MM}{DD}"
    vis_filename = f"VIS_{YYYY}{MM}{DD}{HH}.NC"
    vis_url = f"http://10.148.8.71:7080/thredds/dodsC/cldas/{vis_dir_yyyymmdd}/{vis_filename}"

    return {
        "national":  {"filename": national_filename, "fullpath": national_fullpath},
        "regional":  {"filename": regional_filename, "fullpath": regional_fullpath},
        "cldas_vis": {"filename": vis_filename, "url": vis_url},
    }

In [28]:
time_selected = datetime(2024, 10, 1, 1, 0)
files_selected = build_filenames_and_urls(time_selected)
print(files_selected["national"]["fullpath"])   # \\10.148.44.81\surf\idea\getSurfAutoOrg4Prov\2021\09\SurfAuto_广东_20210901014500.csv
print(files_selected["regional"]["fullpath"])   # \\10.148.44.81\surf\idea\getSurfAwstOrg4Prov\2021\09\SurfAwst_广东_20210901014500.csv
print(files_selected["cldas_vis"]["url"])      # http://10.148.8.71:7080/thredds/dodsC/cldas/20210901/VIS_2021090101.NC

\\10.148.44.81\surf\idea\getSurfAutoOrg4Prov\2024\10\SurfAuto_广东_20241001010000.csv
\\10.148.44.81\surf\idea\getSurfAwstOrg4Prov\2024\10\SurfAwst_广东_20241001010000.csv
http://10.148.8.71:7080/thredds/dodsC/cldas/20241001/VIS_2024100101.NC


In [14]:
def load_visibility_data(data_path='src/visibility_anisotropic_idw.nc'):
    """
    加载能见度数据
    """
    if not data_path.startswith('http') and not path.exists(data_path):
        print(f"错误：找不到文件 {data_path}")
        print("请确保已运行插值程序生成该文件")
        return None
    
    try:
        # 加载数据
        ds = xr.open_dataset(data_path)
        print(f"成功加载数据: {data_path}")
        print(f"数据维度: {ds.dims}")
        print(f"数据变量: {list(ds.data_vars)}")
        
        # 获取能见度数据
        if 'visibility' in ds:
            vis_data = ds['visibility']
        elif 'vis000' in ds:
            vis_data = ds['vis000'][0,0,:,:]
        else:
            # 尝试获取第一个数据变量
            var_name = list(ds.data_vars)[0]
            vis_data = ds[var_name]
            print(f"使用数据变量: {var_name}")
        
        # 转换单位如果需要（从m转换为km）
        if vis_data.max() > 100:  # 如果最大值大于100，可能是米单位
            vis_data = vis_data / 1000
            print("已将单位从米转换为千米")
        
        print(f"能见度数据范围: {vis_data.min().values:.2f} - {vis_data.max().values:.2f} km")
        
        return vis_data
        
    except Exception as e:
        print(f"加载数据时出错: {e}")
        return None

In [15]:
fields = [
    'V01301',    # 站号
    'VF01015_CN',# 站点名称
    'V_CITY',    # 所属地市
    'V_COUNTY',  # 所属县
    'V06001',    # 经度
    'V05001',    # 纬度
    'V07001',    # 海拔
    'V20001',    # 能见度
    'V13003'     # 相对湿度
]
df_nation = pd.read_csv('../data/SurfAuto_20250228000000.csv', encoding='gbk', na_values=9999, usecols=fields)
fields_2 = [
    'V01301',    # 站号
    'VF01015_CN',# 站点名称
    'V_CITY',    # 所属地市
    'V_COUNTY',  # 所属县
    'V06001',    # 经度
    'V05001',    # 纬度
    'V07001',    # 海拔
    'V13003',     # 相对湿度
    'V20001_701_01', # 能见度
]
df_region = pd.read_csv('../data/SurfAwst_20250228000000.csv', encoding='gbk', na_values=9999, usecols=fields_2)
# 重命名字段为英文名
field_map = {
    'V01301': 'code',
    'VF01015_CN': 'name',
    'V_CITY': 'city',
    'V_COUNTY': 'county',
    'V06001': 'lon',
    'V05001': 'lat',
    'V07001': 'altitude',
    'V20001': 'vis',
    'V13003': 'rh',
    'V20001_701_01': 'vis',
}

df_nation = df_nation.rename(columns=field_map)
df_region = df_region.rename(columns=field_map)
# 去除掉df_region中rh为NaN的条目
# df_region = df_region.dropna(subset=['rh', 'county'])
df_region = df_region.dropna(subset=['vis', 'county'])
print(df_nation.head())
print(df_region.count())

    code     lat      lon  altitude  rh   vis       name   county city
0  57988  25.110  113.345     143.2  96   800  乐昌国家基本气象站      乐昌市   韶关
1  57989  25.059  113.763     112.7  96   200  仁化国家基本气象站      仁化县   韶关
2  57996  25.081  114.255     149.7  95  1900  南雄国家基准气候站      南雄市   韶关
3  59071  24.732  112.278     174.3  94  2700  连南国家基本气象站  连南瑶族自治县   清远
4  59072  24.811  112.371     131.7  98   100     连州市气象局      连州市   清远
code        124
lon         124
lat         124
altitude    124
county      124
city        124
name        124
rh           88
vis         124
dtype: int64


In [16]:
df_region

,code,lon,lat,altitude,county,city,name,rh,vis
0,G8337,112.570939,24.482277,311.0,阳山县,清远,许广上行K1180 000M应用气象观测站（交通站）,NaN,1908.0
8,G1275,114.028900,22.095800,45.0,香洲区,珠海,香洲区担杆镇外伶仃流水坑气象观测站,NaN,9952.0
62,G8020,112.216100,23.815028,90.0,怀集县,肇庆,怀集怀城服务区气象观测站,NaN,2314.0
101,G5967,113.579400,22.863600,10.0,东莞市,东莞,东莞沙田镇作业区中路气象观测站,84.0,5109.0
109,G1238,113.421400,22.187500,0.0,香洲区,珠海,香洲区珠海大桥东气象观测站,84.0,9353.0
...,...,...,...,...,...,...,...,...,...
3835,G1281,113.818600,22.158100,7.0,香洲区,珠海,香洲区桂山镇桂山电厂气象观测站,92.0,31331.0
3843,G9576,113.233056,23.117778,8.6,荔湾区,广州,荔湾区多宝街永庆坊,84.0,7166.0
3871,G3133,113.489400,22.907200,12.6,番禺区,广州,番禺区石楼镇清流村,NaN,7116.0
4133,G9533,113.573097,22.816894,4.2,南沙区,广州,南沙区南沙街燃料港口气象观测站,89.0,8758.0


In [21]:
vis_data = load_visibility_data('http://10.148.8.71:7080/thredds/dodsC/cldas/20250228/VIS_2025022800.NC')

成功加载数据: http://10.148.8.71:7080/thredds/dodsC/cldas/20250228/VIS_2025022800.NC
数据维度: FrozenMappingWarningOnValuesAccess({'time': 1, 'level': 1, 'lat': 1201, 'lon': 1401})
数据变量: ['tstr', 'vis000']
已将单位从米转换为千米
能见度数据范围: -0.00 - 65.03 km


In [22]:
print(vis_data)

<xarray.DataArray 'vis000' (lat: 1201, lon: 1401)> Size: 7MB
array([[-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       ...,
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001],
       [-0.001, -0.001, -0.001, ..., -0.001, -0.001, -0.001]],
      shape=(1201, 1401), dtype=float32)
Coordinates:
  * lon      (lon) float32 6kB 70.0 70.05 70.1 70.15 ... 139.9 139.9 139.9 140.0
  * lat      (lat) float32 5kB 0.0 0.05 0.1 0.15 0.2 ... 59.85 59.9 59.95 60.0
    level    float32 4B 1e+03
    time     datetime64[ns] 8B 2025-02-28


In [31]:
def evaluate_visibility_score(vis_data, df_station):
    """
    评估能见度预报效果
    
    参数:
        vis_data: xarray.DataArray, 预报能见度格点数据 (单位: km)
        df_station: pd.DataFrame, 观测站数据，需包含 lon, lat, vis 列 (vis单位: m)
    
    返回:
        results_df: pd.DataFrame, 包含每个站点的对比结果
        stats: dict, 整体统计指标
    """
    # 过滤掉缺失能见度观测值的站点
    df_valid = df_station.dropna(subset=['vis', 'lon', 'lat']).copy()
    
    # 确保观测值单位为km
    df_valid['vis_obs_km'] = df_valid['vis'] / 1000.0
    
    # 初始化结果列表
    forecast_values = []
    obs_values = []
    station_info = []
    
    # 遍历每个站点
    for idx, row in df_valid.iterrows():
        lon_station = row['lon']
        lat_station = row['lat']
        vis_obs = row['vis_obs_km']
        
        # 使用最近邻方法查找格点值
        vis_forecast = vis_data.sel(lon=lon_station, lat=lat_station, method='nearest').values
        
        # 处理负值或异常值（将负值设为0）
        if vis_forecast < 0:
            vis_forecast = 0.0
        
        forecast_values.append(vis_forecast)
        obs_values.append(vis_obs)
        station_info.append({
            'code': row.get('code', ''),
            'name': row.get('name', ''),
            'lon': lon_station,
            'lat': lat_station,
            'vis_obs': vis_obs,
            'vis_forecast': vis_forecast
        })
    
    # 转换为numpy数组进行计算
    forecast_array = np.array(forecast_values)
    obs_array = np.array(obs_values)
    
    # 计算统计指标
    # 1. 误差 (Forecast - Observation)
    error = forecast_array - obs_array
    
    # 2. 绝对误差 (Absolute Error)
    abs_error = np.abs(error)
    
    # 3. 相对误差 (Relative Error, %)
    # 避免除以零，设置一个小阈值
    relative_error = np.where(obs_array > 0.001, 
                              (error / obs_array) * 100, 
                              np.nan)
    
    # 4. 均方误差 (Mean Squared Error)
    mse = np.mean(error ** 2)
    
    # 5. 均方根误差 (Root Mean Squared Error)
    rmse = np.sqrt(mse)
    
    # 6. 平均绝对误差 (Mean Absolute Error)
    mae = np.mean(abs_error)
    
    # 7. 平均偏差 (Mean Bias)
    bias = np.mean(error)
    
    # 8. 标准差
    std_error = np.std(error)
    
    # 9. 相关系数
    correlation = np.corrcoef(forecast_array, obs_array)[0, 1]
    
    # 10. 相对平均绝对误差 (Mean Absolute Percentage Error, MAPE)
    mape = np.nanmean(np.abs(relative_error))
    
    # 构建结果DataFrame
    results_df = pd.DataFrame(station_info)
    results_df['error'] = error
    results_df['abs_error'] = abs_error
    results_df['relative_error'] = relative_error
    
    # 统计指标字典
    stats = {
        'n_stations': len(obs_array),
        'mean_obs': np.mean(obs_array),
        'mean_forecast': np.mean(forecast_array),
        'bias': bias,
        'mae': mae,
        'rmse': rmse,
        'std_error': std_error,
        'correlation': correlation,
        'mape': mape,
        'mse': mse
    }
    
    # 打印统计结果
    print("="*60)
    print("能见度预报评估结果")
    print("="*60)
    print(f"样本数量: {stats['n_stations']}")
    print(f"观测平均值: {stats['mean_obs']:.2f} km")
    print(f"预报平均值: {stats['mean_forecast']:.2f} km")
    print(f"平均偏差 (Bias): {stats['bias']:.2f} km")
    print(f"平均绝对误差 (MAE): {stats['mae']:.2f} km")
    print(f"均方根误差 (RMSE): {stats['rmse']:.2f} km")
    print(f"误差标准差 (Std): {stats['std_error']:.2f} km")
    print(f"相关系数 (R): {stats['correlation']:.4f}")
    print(f"平均相对误差 (MAPE): {stats['mape']:.2f}%")
    print("="*60)
    
    return results_df, stats

In [32]:
# 调用评估函数
results_df, stats = evaluate_visibility_score(vis_data, df_nation)

# 显示部分详细结果
print("\n前10个站点的详细对比:")
print(results_df[['name', 'vis_obs', 'vis_forecast', 'error', 'abs_error', 'relative_error']].head(10))

能见度预报评估结果
样本数量: 87
观测平均值: 4.23 km
预报平均值: 3.04 km
平均偏差 (Bias): -1.18 km
平均绝对误差 (MAE): 1.38 km
均方根误差 (RMSE): 2.69 km
误差标准差 (Std): 2.42 km
相关系数 (R): 0.8393
平均相对误差 (MAPE): 44.28%

前10个站点的详细对比:
        name  vis_obs vis_forecast    error  abs_error  relative_error
0  乐昌国家基本气象站      0.8      0.74971 -0.05029    0.05029       -6.286247
1  仁化国家基本气象站      0.2   0.20763001  0.00763    0.00763        3.815004
2  南雄国家基准气候站      1.9      0.44262 -1.45738    1.45738      -76.704210
3  连南国家基本气象站      2.7       0.9456 -1.75440    1.75440      -64.977779
4     连州市气象局      0.1      0.65052  0.55052    0.55052      550.520027
5     连山县气象局      4.0        1.827 -2.17300    2.17300      -54.324999
6     阳山县气象局      0.1      0.32268  0.22268    0.22268      222.679996
7  乳源国家基本气象站      1.4    1.0576899 -0.34231    0.34231      -24.450721
8  韶关国家基本气象站      6.8       2.2481 -4.55190    4.55190      -66.939705
9  佛冈国家基本气象站      3.3      3.16305 -0.13695    0.13695       -4.150002


In [36]:
def get_vis_score_by_time(time_selected=datetime(2024, 10, 1, 1, 0)):
    """
    根据指定时间，获取能见度预报得分
    """

    files_selected = build_filenames_and_urls(time_selected)
    file_path_nation = files_selected["national"]["fullpath"]   # \\10.148.44.81\surf\idea\getSurfAutoOrg4Prov\2021\09\SurfAuto_广东_20210901014500.csv
    file_path_region = files_selected["regional"]["fullpath"]   # \\10.148.44.81\surf\idea\getSurfAwstOrg4Prov\2021\09\SurfAwst_广东_20210901014500.csv
    url_path_cldas_vis = files_selected["cldas_vis"]["url"]      # http://10.148.8.71:7080/thredds/dodsC/cldas/20210901/VIS_2021090101.NC

    # 读取国家站数据
    df_nation = pd.read_csv(file_path_nation, encoding='gbk', na_values=9999, usecols=fields)
    # 读取区域站数据
    df_region = pd.read_csv(file_path_region, encoding='gbk', na_values=9999, usecols=fields_2)
    df_nation = df_nation.rename(columns=field_map)
    df_region = df_region.rename(columns=field_map)
    df_region = df_region.dropna(subset=['vis', 'county'])

    # 读取能见度预报数据
    vis_data = load_visibility_data(url_path_cldas_vis)

    # 评估能见度预报效果
    results_df, stats = evaluate_visibility_score(vis_data, df_nation)



    return results_df, stats




In [37]:
results_df, stats = get_vis_score_by_time(datetime(2025, 9, 30, 0, 0))
print("\n前10个站点的详细对比:")
print(results_df[['name', 'vis_obs', 'vis_forecast', 'error', 'abs_error', 'relative_error']].head(10))


成功加载数据: http://10.148.8.71:7080/thredds/dodsC/cldas/20250930/VIS_2025093000.NC
数据维度: FrozenMappingWarningOnValuesAccess({'time': 1, 'level': 1, 'lat': 1201, 'lon': 1401})
数据变量: ['tstr', 'vis000']
已将单位从米转换为千米
能见度数据范围: -0.00 - 94.25 km
能见度预报评估结果
样本数量: 87
观测平均值: 19.09 km
预报平均值: 13.68 km
平均偏差 (Bias): -5.41 km
平均绝对误差 (MAE): 6.25 km
均方根误差 (RMSE): 8.06 km
误差标准差 (Std): 5.98 km
相关系数 (R): 0.7777
平均相对误差 (MAPE): 37.73%

前10个站点的详细对比:
        name  vis_obs vis_forecast     error  abs_error  relative_error
0  乐昌国家基本气象站     27.5       21.267  -6.23300    6.23300      -22.665454
1  仁化国家基本气象站      9.5    5.5570498  -3.94295    3.94295      -41.504739
2  南雄国家基准气候站      8.1    5.5058503  -2.59415    2.59415      -32.026539
3  连南国家基本气象站     30.0       9.8405 -20.15950   20.15950      -67.198334
4     连州市气象局      8.5     12.02638   3.52638    3.52638       41.486819
5     连山县气象局     14.3     12.46662  -1.83338    1.83338      -12.820836
6     阳山县气象局      0.3   0.76083004   0.46083    0.46083      153.610015